# UC17 — Cumplimiento Normativo en Conversaciones (Compliance)

Detección automática de riesgos regulatorios en llamadas de agentes de seguros.

**Valor de negocio**: Evitar multas y sanciones regulatorias detectando infracciones proactivamente.

---

## Paso 1: Configuración del Entorno

In [ ]:
CREATE DATABASE IF NOT EXISTS COMPLIANCE_DB;
USE SCHEMA COMPLIANCE_DB.PUBLIC;
CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH WITH WAREHOUSE_SIZE='SMALL' AUTO_SUSPEND=60 AUTO_RESUME=TRUE;
USE WAREHOUSE COMPUTE_WH;

---

## Paso 2: Generar Transcripciones con Cumplimiento Variable

In [ ]:
CREATE OR REPLACE TABLE llamadas_compliance AS
SELECT 'CALL' || LPAD(SEQ4(),5,'0') AS llamada_id,
    'AGT' || LPAD(MOD(SEQ4(),15)+1,3,'0') AS agente_id,
    DATEADD(day,-UNIFORM(1,90,RANDOM()),CURRENT_DATE()) AS fecha,
    CASE MOD(SEQ4(),6)
        WHEN 0 THEN 'Agente: Buenos días, esta llamada será grabada por motivos de calidad. Soy María, ¿en qué puedo ayudarle? Cliente: Quiero información sobre un seguro de vida. Agente: Por supuesto. Le informo de que conforme a la normativa LOSSEAR, tengo obligación de informarle de todas las condiciones antes de contratar. El seguro de vida que ofrecemos tiene una prima anual de 500€ con cobertura de fallecimiento e invalidez. ¿Desea que le envíe la documentación precontractual por email?'
        WHEN 1 THEN 'Agente: Hola, dígame. Cliente: Me interesa un seguro de auto. Agente: Mire, este seguro es el mejor del mercado, le garantizo que no encontrará nada mejor. No necesita leer la letra pequeña, es todo muy sencillo. Solo fírmeme aquí y listo. Se lo activo ahora mismo.'
        WHEN 2 THEN 'Agente: Buenas tardes, le informo de que esta conversación se graba. ¿Me puede confirmar su nombre completo y DNI para verificar su identidad? Cliente: Sí, soy Juan García, DNI 12345678A. Agente: Perfecto. Conforme a la normativa de protección de datos, sus datos personales serán tratados según nuestra política de privacidad. ¿Desea que le envíe una copia?'
        WHEN 3 THEN 'Agente: Hola. Cliente: Quiero reclamar. Me vendieron un seguro por teléfono y me dijeron que cubría todo, pero resulta que la franquicia es de 500€ y nadie me lo dijo. Agente: Déjeme revisar la grabación de la venta... Efectivamente, no veo que se informara de la franquicia durante la venta telefónica. Voy a escalar esto al departamento de reclamaciones.'
        WHEN 4 THEN 'Agente: Buenos días. Cliente: Necesito información sobre el seguro de mi madre, que está enferma. Agente: Claro. ¿Me da el número de póliza? Cliente: Es la 99887. Agente: Veo la póliza. Su madre tiene una cobertura de... Oiga, ¿usted es el titular? Cliente: No, soy su hijo. Agente: No hay problema, le doy toda la información. La cobertura incluye...'
        ELSE 'Agente: Le atiende Carlos. Le informo de la grabación de esta llamada. Cliente: Quiero contratar un seguro de hogar. Agente: Perfecto. Antes de nada, necesito hacerle algunas preguntas para evaluar sus necesidades. Le informo de que tengo deber de asesoramiento conforme a la IDD. Según sus respuestas, le recomendaré el producto que mejor se adapte. No estoy obligado a recomendarle ningún producto específico. ¿Comenzamos?'
    END AS transcripcion
FROM TABLE(GENERATOR(ROWCOUNT=>600));

SELECT llamada_id, agente_id, LEFT(transcripcion,100) AS preview FROM llamadas_compliance LIMIT 10;

---

## Paso 3: Auditoría de Cumplimiento con Cortex AI

Evaluamos 6 criterios regulatorios obligatorios en seguros.

In [ ]:
CREATE OR REPLACE TABLE auditoria_compliance AS
SELECT llamada_id, agente_id, fecha,
    SNOWFLAKE.CORTEX.COMPLETE('mistral-large2',
        'Eres un auditor de cumplimiento normativo de una compañía de seguros española. Evalúa si esta llamada cumple con la normativa. Responde SOLO con JSON: {"aviso_grabacion": true/false, "verificacion_identidad": true/false, "informacion_precontractual": true/false, "proteccion_datos_rgpd": true/false, "promesas_falsas": true/false, "acceso_datos_no_titular": true/false, "nivel_riesgo": "bajo/medio/alto/critico", "infracciones_detectadas": ["infraccion1"], "recomendacion": "máx 20 palabras"}.\n\nTranscripción: ' || transcripcion
    ) AS auditoria_raw
FROM llamadas_compliance;

SELECT llamada_id, agente_id, LEFT(auditoria_raw,400) AS auditoria FROM auditoria_compliance LIMIT 10;

---

## Paso 4: Parsear Resultados y Generar Scoring

In [ ]:
CREATE OR REPLACE TABLE scores_compliance AS
SELECT llamada_id, agente_id, fecha,
    TRY_PARSE_JSON(auditoria_raw) AS a,
    COALESCE(a['aviso_grabacion']::BOOLEAN, FALSE) AS aviso_grabacion,
    COALESCE(a['verificacion_identidad']::BOOLEAN, FALSE) AS verificacion_identidad,
    COALESCE(a['informacion_precontractual']::BOOLEAN, FALSE) AS info_precontractual,
    COALESCE(a['proteccion_datos_rgpd']::BOOLEAN, FALSE) AS rgpd,
    COALESCE(a['promesas_falsas']::BOOLEAN, FALSE) AS promesas_falsas,
    COALESCE(a['acceso_datos_no_titular']::BOOLEAN, FALSE) AS acceso_no_titular,
    COALESCE(a['nivel_riesgo']::VARCHAR, 'medio') AS nivel_riesgo,
    COALESCE(a['recomendacion']::VARCHAR, '') AS recomendacion,
    (CASE WHEN COALESCE(a['aviso_grabacion']::BOOLEAN,FALSE) THEN 1 ELSE 0 END +
     CASE WHEN COALESCE(a['verificacion_identidad']::BOOLEAN,FALSE) THEN 1 ELSE 0 END +
     CASE WHEN COALESCE(a['informacion_precontractual']::BOOLEAN,FALSE) THEN 1 ELSE 0 END +
     CASE WHEN COALESCE(a['proteccion_datos_rgpd']::BOOLEAN,FALSE) THEN 1 ELSE 0 END +
     CASE WHEN NOT COALESCE(a['promesas_falsas']::BOOLEAN,FALSE) THEN 1 ELSE 0 END +
     CASE WHEN NOT COALESCE(a['acceso_datos_no_titular']::BOOLEAN,FALSE) THEN 1 ELSE 0 END
    ) AS criterios_cumplidos
FROM auditoria_compliance;

SELECT nivel_riesgo, COUNT(*) AS llamadas, ROUND(AVG(criterios_cumplidos),1) AS criterios_medio
FROM scores_compliance GROUP BY nivel_riesgo ORDER BY CASE nivel_riesgo WHEN 'critico' THEN 1 WHEN 'alto' THEN 2 WHEN 'medio' THEN 3 ELSE 4 END;

---

## Paso 5: Ranking de Agentes por Compliance

In [ ]:
SELECT agente_id, COUNT(*) AS llamadas,
    ROUND(AVG(criterios_cumplidos),1) AS score_compliance,
    SUM(CASE WHEN nivel_riesgo IN ('alto','critico') THEN 1 ELSE 0 END) AS llamadas_riesgo,
    SUM(CASE WHEN promesas_falsas THEN 1 ELSE 0 END) AS con_promesas_falsas,
    SUM(CASE WHEN acceso_no_titular THEN 1 ELSE 0 END) AS acceso_no_autorizado
FROM scores_compliance GROUP BY agente_id ORDER BY score_compliance;

---

## Paso 6: Dashboard Interactivo

In [ ]:
import streamlit as st
import pandas as pd
from snowflake.snowpark.context import get_active_session

session = get_active_session()
st.title('Cumplimiento Normativo en Conversaciones')
st.markdown('### Compliance Automatizado — Snowflake Cortex')

df = session.table('scores_compliance').to_pandas()

c1,c2,c3,c4 = st.columns(4)
with c1: st.metric('Llamadas Auditadas', len(df))
with c2: st.metric('Riesgo Crítico/Alto', len(df[df['NIVEL_RIESGO'].isin(['critico','alto'])]))
with c3: st.metric('Score Medio', f"{df['CRITERIOS_CUMPLIDOS'].mean():.1f}/6")
with c4: st.metric('Promesas Falsas', df['PROMESAS_FALSAS'].sum())

st.markdown('---')
st.subheader('Distribución por Nivel de Riesgo')
rc = df['NIVEL_RIESGO'].value_counts()
st.bar_chart(pd.DataFrame({'Llamadas': rc.values}, index=rc.index))

st.markdown('---')
st.subheader('Cumplimiento por Criterio')
criterios = pd.DataFrame({'Criterio': ['Aviso Grabación','Verificación ID','Info Precontractual','RGPD','Sin Promesas Falsas','Sin Acceso No Titular'],
    'Cumplimiento %': [df['AVISO_GRABACION'].mean()*100, df['VERIFICACION_IDENTIDAD'].mean()*100, df['INFO_PRECONTRACTUAL'].mean()*100, df['RGPD'].mean()*100, (1-df['PROMESAS_FALSAS'].mean())*100, (1-df['ACCESO_NO_TITULAR'].mean())*100]})
st.bar_chart(criterios.set_index('Criterio'))

st.markdown('---')
st.subheader('Llamadas de Alto Riesgo')
riesgo = df[df['NIVEL_RIESGO'].isin(['critico','alto'])].sort_values('CRITERIOS_CUMPLIDOS')
st.dataframe(riesgo[['LLAMADA_ID','AGENTE_ID','FECHA','NIVEL_RIESGO','CRITERIOS_CUMPLIDOS','PROMESAS_FALSAS','ACCESO_NO_TITULAR','RECOMENDACION']].head(50), use_container_width=True, height=400)
st.caption('Desarrollado con Snowflake Cortex AI + Streamlit')

---

## Paso 7: Limpieza (Opcional)

In [ ]:
-- DROP DATABASE IF EXISTS COMPLIANCE_DB;
-- DROP WAREHOUSE IF EXISTS COMPUTE_WH;